# Extract data from Seurat objects to built .h5mu

.h5mu files are used to distribute and visualize our datasets. We'll extract the matrices and metadata from our dataset and output as .mtx and .csv for use with downstream assembly steps.

In [1]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }
quiet_library(hise)
quiet_library(purrr)
quiet_library(Seurat)

Warning message:
“package ‘purrr’ was built under R version 4.4.3”
Warning message:
“package ‘Seurat’ was built under R version 4.4.3”
Warning message:
“package ‘SeuratObject’ was built under R version 4.4.3”
Warning message:
“package ‘sp’ was built under R version 4.4.3”


In [2]:
if(!dir.exists("output")) {
    dir.create("output")
}
out_files <- c()

In [3]:
file_uuids <- list(
    "cd4_seurat" = "0215838f-38b2-4f72-b94f-c3e4bbb7137d", # CD4 T cell Seurat object
    "cd8_seurat" = "88e3e5cb-e052-4351-be5e-81e9d413d33f" # CD8 T cell Seurat object
)

In [4]:
file_paths <- map(file_uuids, function(uuid) cacheFiles(list(uuid)))

[2026-06-08 17:48:17] INFO  Calling cacheFiles
[1] "downloading fileID 0215838f-38b2-4f72-b94f-c3e4bbb7137d"
[2026-06-08 17:48:22] INFO  Finished cacheFiles (success=TRUE, time_elapsed=3.080s)
[2026-06-08 17:48:22] INFO  Calling cacheFiles
[1] "downloading fileID 88e3e5cb-e052-4351-be5e-81e9d413d33f"
[2026-06-08 17:48:25] INFO  Finished cacheFiles (success=TRUE, time_elapsed=1.905s)


In [5]:
cd4 <- readRDS(file_paths[["cd4_seurat"]])
cd8 <- readRDS(file_paths[["cd8_seurat"]])

In [6]:
names(cd4@meta.data)

[1] "orig.ident"        "nCount_RNA"        "nFeature_RNA"     
 [4] "barcodes"          "adt_umis"          "batch_id"         
 [7] "pool_id"           "chip_id"           "well_id"          
[10] "pbmc_sample_id"    "treatment"         "timepoint"        
[13] "n_reads"           "n_umis"            "n_genes"          
[16] "n_mito_umis"       "frac_mito_umis"    "original_barcodes"
[19] "hto_barcode"       "hto_category"      "Sample"           
[22] "archr_name"        "n_fragments"       "n_unique"         
[25] "n_mito"            "frac_mito"         "RNA_snn_res.0.8"  
[28] "nCount_ADT"        "nFeature_ADT"      "treat_time"       
[31] "CD4pos"            "CD8pos"            "CD4_CD8"

## Extract RNA count matrices

In [7]:
cd4_mat <- cd4@assays$RNA@layers$counts
cd8_mat <- cd8@assays$RNA@layers$counts

In [8]:
str(cd4_mat)

Formal class 'dgCMatrix' [package "Matrix"] with 6 slots
  ..@ i       : int [1:194752686] 24 83 86 113 168 187 190 234 246 247 ...
  ..@ p       : int [1:114547] 0 2157 4291 6382 7942 9809 11532 13071 15561 17280 ...
  ..@ Dim     : int [1:2] 36601 114546
  ..@ Dimnames:List of 2
  .. ..$ : NULL
  .. ..$ : NULL
  ..@ x       : num [1:194752686] 1 1 3 1 1 1 1 1 3 1 ...
  ..@ factors : list()


In [9]:
cd4_mat_file <- "output/cd4_scrna_counts.mtx"
Matrix::writeMM(cd4_mat, cd4_mat_file)
out_files <- c(out_files, cd4_mat_file)

NULL

In [10]:
cd8_mat_file <- "output/cd8_scrna_counts.mtx"
Matrix::writeMM(cd8_mat, cd8_mat_file)
out_files <- c(out_files, cd8_mat_file)

NULL

## Extract RNA features

In [11]:
cd4_feat <- rownames(cd4)
cd8_feat <- rownames(cd8)

In [12]:
cd4_feat_file <- "output/cd4_scrna_feat.csv"
writeLines(c("gene", cd4_feat), cd4_feat_file)
out_files <- c(out_files, cd4_feat_file)

In [13]:
cd8_feat_file <- "output/cd8_scrna_feat.csv"
writeLines(c("gene", cd8_feat), cd8_feat_file)
out_files <- c(out_files, cd8_feat_file)

## Extract ADT matrices

In [14]:
cd4_adt <- cd4@assays$ADT@counts
cd8_adt <- cd8@assays$ADT@counts

In [15]:
str(cd4_adt)

Formal class 'dgCMatrix' [package "Matrix"] with 6 slots
  ..@ i       : int [1:5824075] 1 3 4 5 7 9 10 12 13 14 ...
  ..@ p       : int [1:114547] 0 49 98 148 197 246 293 343 396 445 ...
  ..@ Dim     : int [1:2] 55 114546
  ..@ Dimnames:List of 2
  .. ..$ : chr [1:55] "CD11c" "CD278" "CD11b" "CD16" ...
  .. ..$ : chr [1:114546] "2da9d348fb8111eda35df29f570c0793" "2daec6d2fb8111eda35df29f570c0793" "2db119d2fb8111eda35df29f570c0793" "2db582c4fb8111eda35df29f570c0793" ...
  ..@ x       : num [1:5824075] 4 2 3 147 23 1 2 2 1 4 ...
  ..@ factors : list()


In [16]:
cd4_adt_file <- "output/cd4_adt_counts.mtx"
Matrix::writeMM(cd4_adt, cd4_adt_file)
out_files <- c(out_files, cd4_adt_file)

NULL

In [17]:
cd8_adt_file <- "output/cd8_adt_counts.mtx"
Matrix::writeMM(cd8_adt, cd8_adt_file)
out_files <- c(out_files, cd8_adt_file)

NULL

## Extract ADT features

In [18]:
cd4_feat <- rownames(cd4@assays$ADT@counts)
cd8_feat <- rownames(cd8@assays$ADT@counts)

In [19]:
cd4_feat_file <- "output/cd4_adt_feat.csv"
writeLines(c("feature", cd4_feat), cd4_feat_file)
out_files <- c(out_files, cd4_feat_file)

In [20]:
cd8_feat_file <- "output/cd8_adt_feat.csv"
writeLines(c("feature", cd8_feat), cd8_feat_file)
out_files <- c(out_files, cd8_feat_file)

## Extract cell metadata

In [21]:
cd4_meta <- cd4@meta.data
cd8_meta <- cd8@meta.data

In [22]:
cd4_meta_file <- "output/cd4_metadata.csv"
write.csv(cd4_meta, cd4_meta_file, row.names = FALSE)
out_files <- c(out_files, cd4_meta_file)

In [23]:
cd8_meta_file <- "output/cd8_metadata.csv"
write.csv(cd8_meta, cd8_meta_file, row.names = FALSE)
out_files <- c(out_files, cd8_meta_file)

## Store results in HISE

In [24]:
study_space_uuid <- "40df6403-29f0-4b45-ab7d-f46d420c422e"
title <- "VRd TEA-seq TE component .mtx and .csv with features"

In [25]:
upload_id <- ids::adjective_animal()
upload_id

[1] "visionary_armadillo"

In [26]:
out_list <- as.list(out_files)

In [27]:
out_list

[[1]]
[1] "output/cd4_scrna_counts.mtx"

[[2]]
[1] "output/cd8_scrna_counts.mtx"

[[3]]
[1] "output/cd4_scrna_feat.csv"

[[4]]
[1] "output/cd8_scrna_feat.csv"

[[5]]
[1] "output/cd4_adt_counts.mtx"

[[6]]
[1] "output/cd8_adt_counts.mtx"

[[7]]
[1] "output/cd4_adt_feat.csv"

[[8]]
[1] "output/cd8_adt_feat.csv"

[[9]]
[1] "output/cd4_metadata.csv"

[[10]]
[1] "output/cd8_metadata.csv"

In [28]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/archrpixiv11/.pixi/envs/default/lib/libopenblasp-r0.3.32.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] Seurat_5.4.0       SeuratObject_5.3.0 sp_2.2-1           purrr_1.2.1       
[5] hise_2.16.0       

loaded via a namespace (and not attached):
  [1] bitops_1.0-9           deldir_2.0-4           pbapply_1.7-4         
  [4] gridExtra_2.3          rlang_1.2.0            magrittr_2.0.5        


In [30]:
uploadFiles(
    files = out_list,
    fileTypes = list(
        "Wildcard#ProjectStore", "Wildcard#ProjectStore", "Generic CSV File#ProjectStore", "Generic CSV File#ProjectStore",
        "Wildcard#ProjectStore", "Wildcard#ProjectStore", "Generic CSV File#ProjectStore", "Generic CSV File#ProjectStore",
        "Generic CSV File#ProjectStore", "Generic CSV File#ProjectStore"
    ),
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = unname(file_uuids),
    destination = upload_id
)

Please provide input of comma separated sample ids for the files being uploaded. If you do not have any sample ids, press enter:  


[2026-06-08 18:00:20] INFO  Calling uploadFiles
[2026-06-08 18:00:52] INFO  Finished uploadFiles (success=TRUE, time_elapsed=30.031s)


$Message
[1] "General Okay-ness"

$VisualizationId
[1] "00000000-0000-0000-0000-000000000000"

$AbstractionId
[1] "00000000-0000-0000-0000-000000000000"

$TraceId
[1] "ce95e82c-07e0-4a35-81f4-859ee933ceda"

$ProcessId
[1] "9d3a10f8-7e14-4394-8c18-e67bcf7ad933"

$WorkflowId
[1] "8ba891ac-1875-4dc6-a2d3-81038d085940"

$FileIds
$FileIds[[1]]
[1] "e2c8782f-da0a-4255-9e81-9684da2999cf"

$FileIds[[2]]
[1] "28191894-a932-4c33-9952-8545623d7530"

$FileIds[[3]]
[1] "c8e283d1-991a-4c32-8421-110ffef9063d"

$FileIds[[4]]
[1] "72d0c86e-1370-43cb-91f4-47fc49eee7d9"

$FileIds[[5]]
[1] "50e1e4d6-c596-44bd-bc37-13a1577b6d60"

$FileIds[[6]]
[1] "b79e19dc-d1e2-4967-80b8-9c56f6baccac"

$FileIds[[7]]
[1] "c9d894c5-22e0-45a1-a3ac-f185ad08b47d"

$FileIds[[8]]
[1] "a9de0d17-6a88-45c8-bff2-1403063b71a8"

$FileIds[[9]]
[1] "30a6d419-10a7-4d83-bc3b-2b79d48246fe"

$FileIds[[10]]
[1] "eec01309-7337-43f5-a307-13f60f050b02"